In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен ВТБ за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи ВТБ с MOEX (первые 500 строк)
response_VTBR = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_VTBR = response_VTBR.json()

#извлекаем данные из json
columns_VTBR = data_VTBR['candles']['columns']
rows_VTBR = data_VTBR['candles']['data']

df_VTBR = pd.DataFrame(rows_VTBR, columns=columns_VTBR)
df_VTBR['ticker'] = 'VTBR'

print(f'Count of rows: {len(df_VTBR)}')
#считаем все строки - мало ли их много, а выводит только 500
df_VTBR


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,236.65,247.500,247.500,236.10,8.432204e+08,17351060000,2018-01-03 00:00:00,2018-01-03 23:59:59,VTBR
1,247.50,250.250,251.350,245.55,9.011428e+08,18109880000,2018-01-04 00:00:00,2018-01-04 23:59:59,VTBR
2,251.40,251.000,251.400,247.55,4.691588e+08,9403260000,2018-01-05 00:00:00,2018-01-05 23:59:59,VTBR
3,251.95,254.000,259.350,251.60,1.643983e+09,32082650000,2018-01-09 00:00:00,2018-01-09 23:59:59,VTBR
4,255.00,252.550,256.500,252.00,1.206268e+09,23707190000,2018-01-10 00:00:00,2018-01-10 23:59:59,VTBR
...,...,...,...,...,...,...,...,...,...
495,231.90,232.550,234.925,231.75,1.263601e+09,27047280000,2019-12-16 00:00:00,2019-12-16 23:59:59,VTBR
496,233.10,232.750,233.925,232.10,6.825332e+08,14654050000,2019-12-17 00:00:00,2019-12-17 23:59:59,VTBR
497,232.70,234.600,235.000,230.95,1.028001e+09,22032700000,2019-12-18 00:00:00,2019-12-18 23:59:59,VTBR
498,234.20,232.575,236.075,230.00,1.631040e+09,34847580000,2019-12-19 00:00:00,2019-12-19 23:59:59,VTBR


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции ВТБ в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_VTBR = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_VTBR = response_VTBR.json()
print(f"Count of rows after 500 rows: {len(data_VTBR['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_VTBRnew = data_VTBR['candles']['columns']
rows_VTBRnew = data_VTBR['candles']['data']

df_VTBRnew = pd.DataFrame(rows_VTBRnew, columns=columns_VTBRnew)
df_VTBRnew['ticker'] = 'VTBR'

print(f'Count of rows after 500 rows: {len(df_VTBRnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_VTBR2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_VTBR2 = response_VTBR2.json()
columns_VTBRnew2 = data_VTBR2['candles']['columns']
rows_VTBRnew2 = data_VTBR2['candles']['data']

df_VTBRnew2 = pd.DataFrame(rows_VTBRnew2, columns=columns_VTBRnew2)
df_VTBRnew2['ticker'] = 'VTBR'

print(f'Count of rows after 1000 rows: {len(df_VTBRnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_VTBR3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_VTBR3 = response_VTBR3.json()
columns_VTBRnew3 = data_VTBR3['candles']['columns']
rows_VTBRnew3 = data_VTBR3['candles']['data']

df_VTBRnew3 = pd.DataFrame(rows_VTBRnew3, columns=columns_VTBRnew3)
df_VTBRnew3['ticker'] = 'VTBR'

print(f'Count of rows after 1500 rows: {len(df_VTBRnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_VTBR4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_VTBR4 = response_VTBR4.json()
columns_VTBRnew4 = data_VTBR4['candles']['columns']
rows_VTBRnew4 = data_VTBR4['candles']['data']

df_VTBRnew4 = pd.DataFrame(rows_VTBRnew4, columns=columns_VTBRnew4)
df_VTBRnew4['ticker'] = 'VTBR'

print(f'Count of rows after 2000 rows: {len(df_VTBRnew4)}')

Count of rows after 2000 rows: 69


In [ ]:
#проверяем есть ли данные ниже 2069 дней
logging.info('Запрос 1: проверяем есть ли данные после 2069 строки (start=2069)')
response_VTBR5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2069
      })

data_VTBR5 = response_VTBR5.json()
columns_VTBRnew5 = data_VTBR5['candles']['columns']
rows_VTBRnew5 = data_VTBR5['candles']['data']

df_VTBRnew5 = pd.DataFrame(rows_VTBRnew5, columns=columns_VTBRnew5)
df_VTBRnew5['ticker'] = 'VTBR'

print(f'Count of rows after 2069 rows: {len(df_VTBRnew5)}')

Count of rows after 2069 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_VTBRcandles = pd.concat([df_VTBR, df_VTBRnew, df_VTBRnew2, df_VTBRnew3, df_VTBRnew4], ignore_index=True)
print(f'Total rows: {len(df_VTBRcandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов - {len(df_VTBRcandles)} строк')
df_VTBRcandles

Total rows: 2069


,open,close,high,low,value,volume,begin,end,ticker
0,236.65,247.50,247.50,236.10,8.432204e+08,17351060000,2018-01-03 00:00:00,2018-01-03 23:59:59,VTBR
1,247.50,250.25,251.35,245.55,9.011428e+08,18109880000,2018-01-04 00:00:00,2018-01-04 23:59:59,VTBR
2,251.40,251.00,251.40,247.55,4.691588e+08,9403260000,2018-01-05 00:00:00,2018-01-05 23:59:59,VTBR
3,251.95,254.00,259.35,251.60,1.643983e+09,32082650000,2018-01-09 00:00:00,2018-01-09 23:59:59,VTBR
4,255.00,252.55,256.50,252.00,1.206268e+09,23707190000,2018-01-10 00:00:00,2018-01-10 23:59:59,VTBR
...,...,...,...,...,...,...,...,...,...
2064,71.40,72.92,73.10,71.30,5.230881e+09,72258291,2025-12-26 00:00:00,2025-12-26 23:59:58,VTBR
2065,73.01,72.95,73.15,72.77,6.305460e+08,8640898,2025-12-27 00:00:00,2025-12-27 23:59:57,VTBR
2066,72.83,72.92,73.07,72.60,5.098772e+08,6997767,2025-12-28 00:00:00,2025-12-28 23:59:57,VTBR
2067,73.29,72.17,73.95,71.80,6.656170e+09,91335409,2025-12-29 00:00:00,2025-12-29 23:59:59,VTBR


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2069 строки данных нет. Значит итоговый датафрейм  df_VTBRcandles  содержит полную историю дневных котировок ВТБ за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2069
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **все торговые дни за период** - это все торговые дни ВТБ за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям ВТБ: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги ВТБ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_VTBR_infa = session.get(
    'https://iss.moex.com/iss/securities/VTBR.json')

data_VTBR_infa = response_VTBR_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_VTBR_infa = data_VTBR_infa['description']['columns']
rows_VTBR_infa = data_VTBR_infa['description']['data']

df_VTBR_infa = pd.DataFrame(rows_VTBR_infa, columns=columns_VTBR_infa)
df_VTBR_infa['ticker'] = 'VTBR'

print(f'Count of rows: {len(df_VTBR_infa)}')
df_VTBR_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,VTBR,string,1,0,NaN,VTBR
1,ISSUENAME,Наименование ценной бумаги,Акции обыкновенные,string,2,0,NaN,VTBR
2,NAME,Полное наименование,ао ПАО Банк ВТБ,string,3,0,NaN,VTBR
3,SHORTNAME,Краткое наименование,ВТБ ао,string,4,0,NaN,VTBR
4,ISIN,ISIN код,RU000A0JP5V6,string,5,0,NaN,VTBR
5,REGNUMBER,Номер государственной регистрации,10401000B,string,6,0,NaN,VTBR
6,ISSUESIZE,Объем выпуска,6620418282,number,7,0,0.0,VTBR
7,FACEVALUE,Номинальная стоимость,50,number,8,0,2.0,VTBR
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,VTBR
9,ISSUEDATE,Дата начала торгов,2007-05-28,date,10,0,NaN,VTBR


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_VTBR_infa_best = df_VTBR_infa.set_index('name')['value'].to_frame().T
df_VTBR_infa_best['ticker'] = 'VTBR'
df_VTBR_infa_best = df_VTBR_infa_best.reset_index()

print(f'Полное название: {df_VTBR_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_VTBR_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_VTBR_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_VTBR_infa_best["ISIN"].values[0]}, листинг={df_VTBR_infa_best["LISTLEVEL"].values[0]}')

df_VTBR_infa_best

Полное название: ао ПАО Банк ВТБ
ISIN: RU000A0JP5V6
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,VTBR,Акции обыкновенные,ао ПАО Банк ВТБ,ВТБ ао,RU000A0JP5V6,10401000B,6620418282,50,SUR,...,1,1,1,2006-09-29,Акция обыкновенная,stock_shares,common_share,Акции,269,VTBR


**Результат:**
   
Полное название: ао ПАО Банк ВТБ
ISIN: RU000A0JP5V6
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «ао ПАО Банк ВТБ** - официальное юридическое название компании.
- **ISIN = RU000A0JP5V6** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что ВТБ прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат ВТБ за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов VTBR')
#запрос 3: получаем историю дивидендных выплат ВТБ
#дивиденды влияют на котировки примерно на размер дивиденда
VTBR_diva = session.get(
    'https://iss.moex.com/iss/securities/VTBR/dividends.json')

VTBR_diva_infa = VTBR_diva.json()

#извлекаем данные из json
columns_VTBR_diva = VTBR_diva_infa['dividends']['columns']
rows_VTBR_diva = VTBR_diva_infa['dividends']['data']

df_VTBR_diva = pd.DataFrame(rows_VTBR_diva, columns=columns_VTBR_diva)
df_VTBR_diva['ticker'] = 'VTBR'

print(f'Count of rows: {len(df_VTBR_diva)}')

logging.info(f'Запрос 3: получено {len(df_VTBR_diva)} дивидендных выплат')
df_VTBR_diva

Count of rows: 11


,secid,isin,registryclosedate,value,currencyid,ticker
0,VTBR,RU000A0JP5V6,2014-07-01,0.000000,RUB,VTBR
1,VTBR,RU000A0JP5V6,2015-07-06,0.000000,RUB,VTBR
2,VTBR,RU000A0JP5V6,2016-07-04,0.000000,RUB,VTBR
3,VTBR,RU000A0JP5V6,2016-12-26,0.010000,RUB,VTBR
4,VTBR,RU000A0JP5V6,2017-05-10,0.000000,RUB,VTBR
5,VTBR,RU000A0JP5V6,2018-06-04,0.000000,RUB,VTBR
6,VTBR,RU000A0JP5V6,2019-06-24,0.001099,RUB,VTBR
7,VTBR,RU000A0JP5V6,2020-10-05,0.000773,RUB,VTBR
8,VTBR,RU000A0JP5V6,2021-06-22,0.000017,RUB,VTBR
9,VTBR,RU000A0JP5V6,2021-07-15,0.000017,RUB,VTBR


**Результат:**  Count of rows: 11

API вернул все выплаты дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_VTBR_diva['registryclosedate'] = pd.to_datetime(df_VTBR_diva['registryclosedate'])
df_VTBR_diva_srok = df_VTBR_diva[
    (df_VTBR_diva['registryclosedate'] >= '2018-01-01') &
    (df_VTBR_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_VTBR_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_VTBR_infa_best["ISIN"].values[0]}, листинг={df_VTBR_infa_best["LISTLEVEL"].values[0]}')
df_VTBR_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 6


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,5,VTBR,RU000A0JP5V6,2018-06-04,0.000000,RUB,VTBR
1,6,VTBR,RU000A0JP5V6,2019-06-24,0.001099,RUB,VTBR
2,7,VTBR,RU000A0JP5V6,2020-10-05,0.000773,RUB,VTBR
3,8,VTBR,RU000A0JP5V6,2021-06-22,0.000017,RUB,VTBR
4,9,VTBR,RU000A0JP5V6,2021-07-15,0.000017,RUB,VTBR
5,10,VTBR,RU000A0JP5V6,2025-07-11,25.580000,RUB,VTBR


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 6


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по ВТБ - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные VTBR')
#запрос 4: получаем текущие торговые данные ВТБ
response_VTBR_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/VTBR.json')

data_VTBR_markdata = response_VTBR_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_VTBR_markdata = data_VTBR_markdata['marketdata']['columns']
rows_VTBR_markdata = data_VTBR_markdata['marketdata']['data']

df_VTBR_markdata = pd.DataFrame(rows_VTBR_markdata, columns=columns_VTBR_markdata)
df_VTBR_markdata['ticker'] = 'VTBR'

print(f'Count of rows: {len(df_VTBR_markdata)}')

logging.info(f'Запрос 4: получено {len(df_VTBR_markdata)} строк market data')
df_VTBR_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,VTBR,TQBR,86.045,None,86.06,None,0.015,5741561,18074247,86.45,...,2026-03-19 23:20:00,85.695,255752,5.697532e+11,23:20:00,None,2995195096,2,-2.548861e+09,VTBR


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_VTBR_markdata.columns]

df_VTBR_markdata_best = df_VTBR_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена VTBR: {df_VTBR_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_VTBR_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_VTBR_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена VTBR={df_VTBR_markdata_best["LAST"].values[0]}')

df_VTBR_markdata_best

Последняя цена VTBR: 86.06
Изменение к пред. закрытию: -0.45
Капитализация: 569753197348.92


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,VTBR,86.06,-0.45,86.045,86.06,86.45,86.71,85.5,86.12,99327,34783388,2995195096,5.697532e+11,23:05:00


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена VTBR: 86.06
Изменение к пред. закрытию: -0.45
Капитализация: 569753197348.92
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У ВТБ это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности ВТБ на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит ВТБ.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы VTBR')
#запрос 5: получаем список биржевых индексов в которые входит ВТБ
response_VTBR_idishka = session.get(
    'https://iss.moex.com/iss/securities/VTBR/indices.json')

data_VTBR_idishka = response_VTBR_idishka.json()

#извлекаем данные из json
columns_VTBR_idishka = data_VTBR_idishka['indices']['columns']
rows_VTBR_idishka = data_VTBR_idishka['indices']['data']

df_VTBR_idishka = pd.DataFrame(rows_VTBR_idishka, columns=columns_VTBR_idishka)
df_VTBR_idishka['ticker'] = 'VTBR'

print(f'Count of rows: {len(df_VTBR_idishka)}')

logging.info(f'Запрос 5: VTBR входит в {len(df_VTBR_idishka)} индексов')

df_VTBR_idishka

Count of rows: 17


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2013-06-18,2026-03-18,1665.38,0.00,-0.02,VTBR
1,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,VTBR
2,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,VTBR
3,MOEXBC,Индекс голубых фишек,2010-01-14,2026-03-19,19057.56,-0.03,-5.43,VTBR
4,MRRT,Индекс MRRT,2021-01-15,2026-03-18,2335.80,0.05,1.15,VTBR
5,MRSV,Индекс MRSV,2021-01-15,2026-03-18,2173.86,-0.22,-4.80,VTBR
6,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2021-03-25,2026-03-18,2237.45,-0.22,-4.93,VTBR
7,RTSfn,Индекс РТС финансов,2009-09-30,2026-03-19,199.90,-2.59,-5.32,VTBR
8,RTSI,Индекс РТС,2009-09-30,2026-03-19,1065.32,-2.11,-22.97,VTBR
9,RUBMI,Индекс РТС широкого рынка,2013-05-20,2026-03-19,778.15,-2.12,-16.89,VTBR


**Результат:**  Count of rows: 17


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли VTBR в IMOEX (главный российский фондовый индекс)
if len(df_VTBR_idishka) > 0:
    imoex_check = df_VTBR_idishka[df_VTBR_idishka['SECID'] == 'IMOEX']
    print(f'VTBR входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('VTBR не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по VTBR собраны')


VTBR входит в IMOEX: True


**Результат:**
   
VTBR входит в IMOEX: True
   

ВТБ входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** ВТБ входит в IMOEX и другие 17 индексов Мосбиржи. В EDA это будет важным признаком при анализе силы реакции на новости.



